In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import pearsonr # Per calcolare la correlazione
from scipy.stats import linregress # Per la linea di regressione

# Caricamento
df = pd.read_csv('./analysis/esecuzione_20250720_181206/performance_summary_1753028337.csv')

# Manipolazioni varie
df['Permutazione'] = df['File'].str.extract(r'(perm_\d{2})')
df.drop(columns=['File'], axis=1, inplace=True)
df.sort_values(by='Permutazione', inplace=True)
df = df[['Permutazione'] + [col for col in df.columns if col != 'Permutazione']]
df = df.reset_index(drop=True)

def generate_stability_analysis_plots_plotly(df: pd.DataFrame):
    """
    Genera un set di quattro grafici Plotly per l'analisi della stabilità dell'algoritmo.
    """

    mean_time = df['Tempo_Esecuzione_s'].mean()
    std_time = df['Tempo_Esecuzione_s'].std()
    cv_time = std_time / mean_time if mean_time != 0 else np.nan
    median_time = df['Tempo_Esecuzione_s'].median()
    mean_ipotesi = df['Ipotesi_Generate'].mean()
    std_ipotesi = df['Ipotesi_Generate'].std()
    cv_ipotesi = std_ipotesi / mean_ipotesi if mean_ipotesi != 0 else np.nan

    correlation_coeff, _ = pearsonr(df['Ipotesi_Generate'], df['Tempo_Esecuzione_s'])
    
    slope, intercept, r_value, p_value, std_err = linregress(df['Ipotesi_Generate'], df['Tempo_Esecuzione_s'])
    
    # Creazione della griglia di subplot (2x2)
    fig = make_subplots(rows=2, cols=2,
                        subplot_titles=(
                            "Distribuzione Tempi di Esecuzione",
                            "Distribuzione Frequenza Tempi",
                            "Relazione Complessità-Tempo",
                            "Stabilità Metriche (CV)"
                        ),
                        horizontal_spacing=0.1,
                        vertical_spacing=0.15)

    # --- Grafico 1: Box Plot (Tempo di Esecuzione) ---
    box_plot = px.box(df, y='Tempo_Esecuzione_s',
                      title=f'Media: {mean_time:.3f}s<br>Std: {std_time:.3f}s<br>CV: {cv_time:.3f}')
    
    for trace in box_plot.data:
        fig.add_trace(trace, row=1, col=1)
    fig.update_yaxes(title_text='Tempo di Esecuzione (s)', row=1, col=1)
    fig.update_xaxes(title_text='', showticklabels=False, row=1, col=1)

    # --- Grafico 2: Istogramma (Distribuzione Frequenza Tempi) ---
    hist_plot = px.histogram(df, x='Tempo_Esecuzione_s', nbins=10, opacity=0.7,
                             labels={'Tempo_Esecuzione_s': 'Tempo di Esecuzione (s)'},
                             color_discrete_sequence=['lightblue'])

    for trace in hist_plot.data:
        fig.add_trace(trace, row=1, col=2)
    
    fig.add_vline(x=mean_time, line_dash="solid", line_color="green", line_width=2,
                  annotation_text=f"Media: {mean_time:.3f}s", annotation_position="right",
                  annotation_font_color="green", annotation_font_size=10,
                  row=1, col=2) # Specifica il subplot

    fig.add_vline(x=median_time, line_dash="dash", line_color="red", line_width=2,
                  annotation_text=f"Mediana: {median_time:.3f}s", annotation_position="left",
                  annotation_font_color="red", annotation_font_size=10,
                  row=1, col=2)
    # fig.update_layout(legend=dict(x=0.8, y=0.9, xanchor='right', yanchor='top', tracegroupgap=0), row=1, col=2)


    # --- Grafico 3: Scatter Plot (Relazione Complessità-Tempo) ---
    scatter_plot = px.scatter(df, x='Ipotesi_Generate', y='Tempo_Esecuzione_s',
                              labels={'Ipotesi_Generate': 'Ipotesi Generate', 'Tempo_Esecuzione_s': 'Tempo di Esecuzione (s)'},
                              color_discrete_sequence=['orange']) # Colore come nell'immagine

    for trace in scatter_plot.data:
        fig.add_trace(trace, row=2, col=1)

    x_range = np.array([df['Ipotesi_Generate'].min(), df['Ipotesi_Generate'].max()])
    y_reg = slope * x_range + intercept
    fig.add_trace(go.Scatter(x=x_range, y=y_reg, mode='lines', line=dict(color='red', width=2),
                             name='Regressione Lineare'),
                  row=2, col=1)
    
    fig.add_annotation(dict(xref='x2 domain', yref='y2 domain',
                            x=0.1, y=0.9, # Posizione nel grafico (dominio 0-1)
                            text=f'Correlazione: {correlation_coeff:.3f}',
                            showarrow=False,
                            bgcolor='rgba(255, 255, 255, 0.8)',
                            bordercolor='black', borderwidth=0.5,
                            font=dict(size=10)),
                        row=2, col=1) # Specifica il subplot

    fig.update_xaxes(title_text='Ipotesi Generate', row=2, col=1)
    fig.update_yaxes(title_text='Tempo di Esecuzione (s)', row=2, col=1)

    fig.data[-1].showlegend = False


    # --- Grafico 4: Bar Plot (Stabilità Metriche - CV) ---
    cv_df = pd.DataFrame({
        'Metrica': ['Tempo', 'Ipotesi'],
        'CV_Value': [cv_time, cv_ipotesi]
    })
    
    bar_cv_plot = px.bar(cv_df, x='Metrica', y='CV_Value',
                         color='Metrica', 
                         color_discrete_map={'Tempo': 'orange', 'Ipotesi': 'green'}, 
                         labels={'CV_Value': 'Coefficiente di Variazione (CV)'})

    for trace in bar_cv_plot.data:
        fig.add_trace(trace, row=2, col=2)
    
    soglia_moderata = 0.15
    soglia_instabilita = 0.30

    fig.add_hline(y=soglia_moderata, line_dash="dash", line_color="orange", opacity=0.7,
                  annotation_text="Soglia moderata", annotation_position="top right",
                  annotation_font_size=10, annotation_font_color="orange",
                  row=2, col=2)
    
    fig.add_hline(y=soglia_instabilita, line_dash="dash", line_color="red", opacity=0.7,
                  annotation_text="Soglia instabilità", annotation_position="top right",
                  annotation_font_size=10, annotation_font_color="red",
                  row=2, col=2)
    
    for i, row in cv_df.iterrows():
        fig.add_annotation(dict(x=row['Metrica'], y=row['CV_Value'],
                                text=f"{row['CV_Value']:.3f}",
                                showarrow=False,
                                yshift=10, 
                                font=dict(size=10, color='black')),
                            row=2, col=2)

    fig.update_xaxes(title_text='', row=2, col=2) 
    fig.update_yaxes(title_text='Coefficiente di Variazione', row=2, col=2, range=[0, 0.35]) 

    fig.update_layout(title_text='Analisi Stabilità Algoritmo MHS',
                      title_x=0.5, 
                      height=1000, 
                      width=1200, 
                      showlegend=True,
                      hovermode="closest",
                      template="plotly_white",
                      margin=dict(l=50, r=50, t=80, b=50)
                     )

    fig.show()


generate_stability_analysis_plots_plotly(df)

In [2]:
mean_tempo = df['Tempo_Esecuzione_s'].mean()

# Crea il grafico a barre iniziale con Plotly Express
fig_tempo = px.bar(df,
                   x='Permutazione',
                   y='Tempo_Esecuzione_s',
                   title='Tempo di Esecuzione per Permutazione',
                   labels={'Tempo_Esecuzione_s': 'Tempo (secondi)'},
                   hover_data=['Picco_Memoria_MB', 'Ipotesi_Generate'])

# Aggiungi una linea orizzontale per la media
fig_tempo.add_hline(y=mean_tempo,
                    line_dash="dash",
                    line_color="red",
                    annotation_text=f"Media: {mean_tempo:.3f} s",
                    annotation_position="bottom right")

fig_tempo.show()

fig_tempo_line = px.line(df,
                         x='Permutazione',
                         y='Tempo_Esecuzione_s',
                         title='Tempo di Esecuzione per Permutazione',
                         labels={'Tempo_Esecuzione_s': 'Tempo (secondi)'},
                         hover_data=['Picco_Memoria_MB', 'Ipotesi_Generate'],
                         markers=True) # Aggiunge un marker per ogni punto dati

# Aggiungi una linea orizzontale per la media
fig_tempo_line.add_hline(y=mean_tempo,
                        line_dash="dash",
                        line_color="red",
                        annotation_text=f"Media: {mean_tempo:.3f} s",
                        annotation_position="bottom right")

fig_tempo_line.show()

# Calcola la media del Picco di Memoria
mean_memoria = df['Picco_Memoria_MB'].mean()

# Crea il grafico a barre iniziale con Plotly Express
fig_memoria = px.bar(df,
                     x='Permutazione',
                     y='Picco_Memoria_MB',
                     title='Picco di Memoria per Permutazione',
                     labels={'Picco_Memoria_MB': 'Memoria (MB)'},
                     hover_data=['Tempo_Esecuzione_s', 'Ipotesi_Generate'])

# Aggiungi una linea orizzontale per la media
fig_memoria.add_hline(y=mean_memoria,
                      line_dash="dash",
                      line_color="red",
                      annotation_text=f"Media: {mean_memoria:.1f} MB",
                      annotation_position="bottom right")

fig_memoria.show()

fig_memoria_line = px.line(df,
                          x='Permutazione',
                          y='Picco_Memoria_MB',
                          title='Picco di Memoria per Permutazione',
                          labels={'Picco_Memoria_MB': 'Memoria (MB)'},
                          hover_data=['Tempo_Esecuzione_s', 'Ipotesi_Generate'],
                          markers=True)

# Aggiungi una linea orizzontale per la media
fig_memoria_line.add_hline(y=mean_memoria,
                          line_dash="dash",
                          line_color="red",
                          annotation_text=f"Media: {mean_memoria:.1f} MB",
                          annotation_position="bottom right")

fig_memoria_line.show()

mean_cpu = df['CPU_Media_%'].mean()

# Crea il grafico a barre iniziale con Plotly Express
fig_cpu = px.bar(df,
                 x='Permutazione',
                 y='CPU_Media_%',
                 title='Utilizzo CPU per Permutazione',
                 labels={'CPU_Media_%': 'Utilizzo CPU (%)'},
                 hover_data=['Tempo_Esecuzione_s', 'Ipotesi_Generate'])

# Aggiungi una linea orizzontale per la media
fig_cpu.add_hline(y=mean_cpu,
                  line_dash="dash",
                  line_color="red",
                  annotation_text=f"Media: {mean_cpu:.1f} %",
                  annotation_position="bottom right")

fig_cpu.show()

fig_cpu_line = px.line(df,
                      x='Permutazione',
                      y='CPU_Media_%',
                      title='Utilizzo CPU per Permutazione',
                      labels={'CPU_Media_%': 'Utilizzo CPU (%)'},
                      hover_data=['Tempo_Esecuzione_s', 'Ipotesi_Generate'],
                      markers=True)

# Aggiungi una linea orizzontale per la media
fig_cpu_line.add_hline(y=mean_cpu,
                      line_dash="dash",
                      line_color="red",
                      annotation_text=f"Media: {mean_cpu:.1f} %",
                      annotation_position="bottom right")

fig_cpu_line.show()

In [3]:
fig = px.scatter(df,
                 x='Ipotesi_Generate',
                 y='Tempo_Esecuzione_s',
                 title='Tempo di Esecuzione vs Ipotesi Generate',
                 labels={'Tempo_Esecuzione_s': 'Tempo (secondi)'},
                 hover_name='Permutazione', # Il nome del Permutazione appare in grassetto nel tooltip
                 color='Picco_Memoria_MB', # Colora i punti in base al Permutazione (utile con più Permutazione)
                 size='Picco_Memoria_MB' # La dimensione del punto indica la memoria usata
                )
fig.show()

fig = px.density_contour(df, 
                         x="Ipotesi_Generate", 
                         y="Tempo_Esecuzione_s", 
                         marginal_x="histogram", 
                         marginal_y="histogram")
fig.show()

In [4]:
fig = px.scatter(df,
                 x='Max_Livello_Size',
                 y='Picco_Memoria_MB',
                 title='Picco di Memoria vs Max Livello Size',
                 labels={'Picco_Memoria_MB': 'Memoria (MB)'},
                 hover_name='Permutazione',
                 color='Tempo_Esecuzione_s', # Colora i punti in base al Tempo di Esecuzione
                 size='Tempo_Esecuzione_s' # La dimensione del punto indica il tempo
                )
fig.show()

In [5]:
numeric_cols = ['Tempo_Esecuzione_s', 'Picco_Memoria_MB', 'MHS_Trovati', 'Ipotesi_Generate', 'Livelli_Esplorati', 'Max_Livello_Size']

fig = px.scatter_matrix(df,
                        dimensions=numeric_cols,
                        title='Matrice di Scatter Plot delle Metriche di Performance',
                        height=1400,
                        color='Tempo_Esecuzione_s'
                       )
fig.update_traces(diagonal_visible=False)
fig.show()

In [6]:
df

,Permutazione,Tempo_Esecuzione_s,Picco_Memoria_MB,CPU_Media_%,MHS_Trovati,Ipotesi_Generate,Livelli_Esplorati,Max_Livello_Size,Complessità_Temporale,Complessità_Spaziale
0,perm_00,27.666,68.3,99.7,445,1568,2,1512,O(n^k) - polynomial,O(n) - linear
1,perm_01,26.608,67.3,99.6,445,1568,2,1512,O(n^k) - polynomial,O(n) - linear
2,perm_02,16.305,65.8,99.3,445,1504,2,1448,O(n^k) - polynomial,O(n) - linear
3,perm_03,17.669,68.3,99.3,445,1453,2,1397,O(n^k) - polynomial,O(n) - linear
4,perm_04,27.112,67.3,99.6,445,1568,2,1512,O(n^k) - polynomial,O(n) - linear
5,perm_05,17.814,68.4,99.5,445,1349,2,1293,O(n^k) - polynomial,O(n) - linear
6,perm_06,20.292,68.3,99.6,445,1468,2,1412,O(n^k) - polynomial,O(n) - linear
7,perm_07,21.534,67.3,99.5,445,1385,2,1329,O(n^k) - polynomial,O(n) - linear
8,perm_08,19.110,68.3,99.4,445,1455,2,1399,O(n^k) - polynomial,O(n) - linear
9,perm_09,19.571,68.4,99.5,445,1443,2,1387,O(n^k) - polynomial,O(n) - linear


In [7]:
# Lista delle categorie/assi (radials)
categories = ['Tempo_Esecuzione_s', 'Picco_Memoria_MB', 'CPU_Media_%'] # Ordine in cui verranno visualizzati gli assi

data_for_range = df[categories]

# Trova il valore minimo tra tutti i dati delle categorie
min_val = data_for_range.min().min()
# Trova il valore massimo tra tutti i dati delle categorie
max_val = data_for_range.max().max()

# Aggiungi un piccolo "padding" (margine) per non far toccare i dati il bordo
# Puoi aggiustare questo fattore a seconda di quanto margine vuoi.
# Per il minimo, se min_val è vicino a 0, potresti volerlo impostare a 0.
# Se min_val è grande (es. 1000), un padding del 5-10% in meno ha senso.
padding_factor_min = 0.9 # Riduce il minimo di un 10% per un piccolo spazio sotto
padding_factor_max = 1.1 # Aumenta il massimo di un 10% per un piccolo spazio sopra

# Calcola il range effettivo dell'asse radiale
# Assicurati che il minimo non vada sotto zero se i tuoi dati sono sempre positivi
# Esempio: min_val_padded = max(0, min_val * padding_factor_min)
# Un approccio comune è impostare il minimo a 0 o al minimo effettivo meno un piccolo delta.
# Per un range dinamico che parte dal basso senza troppo spreco:
# Se min_val è > 0, possiamo iniziare l'asse da un valore leggermente inferiore al min_val
start_range = min_val - (max_val - min_val) * 0.05 # Sottrai 5% del range totale
if start_range < 0:
    start_range = 0 # Evita range negativi se i dati non li contemplano

end_range = max_val + (max_val - min_val) * 0.05 # Aggiungi 5% del range totale

fig = go.Figure()

for index, row in df.iterrows():
    entity_name = row['Permutazione']
    values = [row[cat] for cat in categories]

    # Per chiudere la forma, il primo valore deve essere ripetuto alla fine
    values.append(values[0])
    categories_closed = categories + [categories[0]]

    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=categories_closed,
        fill='toself',
        name=entity_name,
        text=[f'{v:.3f}' for v in values[:-1]] + [''],
        hoverinfo='text+name',
        mode='lines+markers+text',
        textposition="top center",
        showlegend=True
    ))

# Aggiorna il layout per lo stile del grafico a ragnatela
fig.update_layout(
    polar=dict(
        radialaxis_visible=True,
        radialaxis=dict(
            # *** QUI IMPOSTIAMO IL RANGE OTTIMIZZATO ***
            range=[start_range, end_range],
            showticklabels=True,
            # Per i tick, puoi continuare a usare np.linspace o lasciare che Plotly decida automaticamente
            # np.linspace(start_range, end_range, 5) per 5 tick equidistanti
            tickformat=".1f", # Formato delle etichette (es. 0.0, 0.5, 1.0)
            linecolor='grey',
            linewidth=1,
            gridcolor='lightgrey',
            gridwidth=1
        ),
        angularaxis=dict(
            direction="clockwise",
            tickvals=list(range(len(categories))),
            ticktext=categories,
            linewidth=1,
            linecolor='grey',
            rotation=90
        )
    ),
    title_text='Spider Chart - Mem, CPU Usage and Elapsed Time',
    showlegend=True,
    legend=dict(
        x=0.85,
        y=0.99,
        bgcolor='rgba(255, 255, 255, 0.5)',
        bordercolor='rgba(0, 0, 0, 0)'
    )
)

fig.show()


In [8]:
fig = px.scatter_ternary(df, a="Tempo_Esecuzione_s", b="CPU_Media_%", c="Picco_Memoria_MB", hover_name="Permutazione",
    color="Tempo_Esecuzione_s", size="Tempo_Esecuzione_s", size_max=15,
    # color_discrete_map = {"Joly": "blue", "Bergeron": "green", "Coderre":"red"} 
    )
fig.show()